# Agentic AI: step by step

**Part A: Setup.** Run these cells first in every new Colab session. Colab forgets everything when a session ends, so we rebuild the setup each time.



```powershell
cd D:\parham\Parham_Learning\agentic-ai
Rename-Item stage-00-colab-check.ipynb stage-00-colab-scratch.ipynb
Set-Content -Path 01-agent-basics.ipynb -Value '{"cells":[],"metadata":{},"nbformat":4,"nbformat_minor":5}'
code 01-agent-basics.ipynb
```

### Cell 1: Check the GPU
Confirms Colab gave us a GPU. We want `Tesla T4`.

In [2]:
# Cell 1: Confirm Colab gave us a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### Cell 2: Install helper tools
`zstd` unpacks Ollama's files. `pciutils` and `lshw` let the installer find the GPU. We install them first so the GPU is detected on the first try.

In [3]:
# Cell 2: Install zstd (unpacking) and pciutils + lshw (GPU detection)
!apt-get install -y -q zstd pciutils lshw

Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
pciutils is already the newest version (1:3.10.0-2build1).
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


### Cell 3: Install Ollama
Installs the Ollama program. Look for "NVIDIA GPU installed" near the end. The "systemd is not running" warning is normal in Colab.

In [4]:
# Cell 3: Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


### Cell 4: Start the server
The server is the program that waits for requests. It must be running before Python can talk to the model. We run it in the background and save its messages to `ollama.log`.

In [5]:
# Cell 4: Start the Ollama server in the background
!nohup ollama serve > ollama.log 2>&1 &
!sleep 8

### Cell 5: Check the server found the GPU
Searches the server log for GPU lines. We want `Tesla T4` and about 14 GiB available.

In [6]:
# Cell 5: Check the server log for GPU detection
!grep -i -E "gpu|cuda|nvidia" ollama.log | head -n 10

### Cell 6: Install the Python client
The `ollama` package lets Python send messages to the server.

In [7]:
# Cell 6: Install the Ollama Python client
!pip install -q ollama

### Cell 7: Download the model
Downloads Gemma 4 (small size) onto Colab's disk. We use Python so the output stays clean. A reply of `success` means it worked.

In [8]:
# Cell 7: Download the model through the Python client
import ollama

print(ollama.pull("gemma4:e2b").status)

success


### Cell 8: First message
A quick test that the whole setup works. The first call is slow because the model loads onto the GPU.

In [9]:
# Cell 8: Send one message and print the reply
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
)
print(response.message.content)

Hello! I hope you're having a wonderful day.


## Part B: Stage 1, talking to the model from Python

The call stays the same. Only the message text changes. We try four different messages and compare.

### Cell 9: Ask a plain question
The simplest use. We send one question and print the reply.

In [10]:
# Cell 9: Ask a plain question
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "What is a Bill of Materials (BOM) in two sentences?"}],
)

print(response.message.content)

A Bill of Materials (BOM) is a comprehensive list of all the raw materials, sub-assemblies, parts, and components required to manufacture a finished product. It is essential for tracking inventory, calculating production costs, managing procurement, and ensuring the efficient assembly of the final item.


### Cell 10: Control the length
The model follows instructions about size. We ask for exactly one sentence, then compare with Cell 9.

In [11]:
# Cell 10: Ask for a very short answer

response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "In one short sentence, what is a Bill of Materials (BOM)?"}],)
print(response.message.content)

A Bill of Materials (BOM) is a comprehensive list of all the raw materials, components, and parts required to manufacture a finished product.


### Cell 11: Control the style
We explain the same idea to a different reader. The facts stay the same, and the wording changes.

In [12]:
# Cell 11: Explain the same idea to a child

response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "Explain what a Bill of Materials (BOM) is to a child."}],)
print(response.message.content)

Imagine you want to build the coolest LEGO spaceship ever!

A **Bill of Materials** (we can call it the "BOM" for short) is like a super important **shopping list** or a **secret instruction sheet** that tells you *exactly* everything you need to build something.

### Here is how it works:

**1. It's a List:**
The BOM is just a long list of every single piece of stuff needed.

*   For your LEGO spaceship, the BOM would say:
    *   3 big blue bricks
    *   1 red wing piece
    *   4 tiny wheels
    *   1 piece of sparkly silver for the antenna

**2. It Stops You From Forgetting Things:**
If you just started building without a list, you might accidentally forget the tiny wheels! The BOM makes sure you have every single piece before you start. It keeps you organized!

**3. It Tells You Exactly How Much:**
Sometimes the list doesn't just say "bricks"—it says "3 big blue bricks." It tells you the exact count so you don't run out halfway through!

### Think of it like this:

*   **If you a

### Cell 12: Control the format
We ask for a table. Later, the same idea will let us ask for structured data that code can read.

In [13]:
# Cell 12: Ask for a markdown table
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "Make a markdown table of 3 parts in a simple desk lamp: part name, material, and quantity."}],
)
print(response.message.content)

| Part Name | Material | Quantity |
| :--- | :--- | :--- |
| Lamp Base | Metal or Plastic | 1 |
| Lamp Arm/Neck | Metal or Plastic | 1 |
| Lampshade | Fabric or Plastic | 1 |


### Cell 13: Improve a message
Earlier, the model skipped a pronunciation. A clearer instruction gives a more complete answer.

In [14]:
# Cell 13: Ask for hello in four languages with a pronunciation for every word

response = ollama.chat(
    model="gemma4:e2b",
    messages = [{"role": "user", "content": (
        "Say hello in Georgian, Farsi, Welsh and Japanese. "
        "Use one greeting per language. "
        "Give the pronunciation of every greeting in parentheses."
    )}]
)
print(response.message.content)


**Georgian:** გამარჯობა (ga-mar-jo-ba)

**Farsi:** سلام (salām)

**Welsh:** Helô (hel-oh)

**Japanese:** こんにちは (konnichiwa)


## Stage 1 result: the message controls the answer

| Cell | What we changed | Result |
|---|---|---|
| 9 | A plain question | A normal answer |
| 10 | Asked for one short sentence | A shorter answer |
| 11 | Asked for a child's level | Simpler wording |
| 12 | Asked for a table | A table |
| 13 | Added clearer rules | A more complete answer |

**Lesson:** the model does what the message says. Clear instructions give better answers.

## Part C: Stage 2, the model has no memory

Each call to `ollama.chat` starts fresh. We test this with two separate calls.

### Cell 15: Call one, tell the model a fact

In [15]:
first = ollama.chat(
    model = "gemma4:e2b",
    messages = [{"role": "user", "content": "My favorite color is green."}]
)

print(first.message.content)

Green is a wonderful color! It's often associated with nature, growth, freshness, and tranquility.

What is it about the color green that you like the most?


### Cell 16: Call two, ask about the fact
This is a new call. We send only the new question.

In [16]:
# Cell 16: Second call - ask about the fact from the first call
second = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": "What is my favourite colour?"}],
)
print(second.message.content)

I don't have access to your personal information, so I don't know what your favorite color is.

You'll have to tell me! 😊


In [17]:
# Cell 17: Inspect what goes in and what comes out
sent = [{"role": "user", "content": "Say hi in three words."}]

response = ollama.chat(model="gemma4:e2b", messages=sent)

print(type(sent))              # what we sent: a list
print(len(sent))               # how many messages in the list
print(type(response.message))  # what came back: one Message object
print(response.message.role)   # who wrote it
print(response.message.content)  # the text

<class 'list'>
1
<class 'ollama._types.Message'>
assistant
Hi there!


## Messages and roles

| Term | Meaning |
|---|---|
| `messages` | The list we send in. It holds the whole conversation. |
| `response.message` | One message coming back: the model's reply. |
| `role` | Who wrote the message. |
| `content` | The text of the message. |

| Role | Written by | Purpose |
|---|---|---|
| `system` | Developer | Standing rules for the whole chat |
| `user` | Person | Questions and instructions |
| `assistant` | Model | Its earlier replies |
| `tool` | Our code | Results of tool calls (explained in Stage 5) |

In [18]:
# Cell 18: Use a system message to set the rules
reply = ollama.chat(
    model="gemma4:e2b",
    messages=[
        {"role": "system", "content": "You are a strict assistant. Answer in 5 words or fewer."},
        {"role": "user", "content": "What is a Bill of Materials?"},
    ],
)
print(reply.message.content)

List of required parts for a product.


In [19]:
# Cell 19: Count the words in the reply
print(len(reply.message.content.split()))

7


### Cell 20: Send the full conversation
We put all three messages in one list: what we said, what the model replied, and the new question.

In [20]:
# Cell 20: Send the full conversation, not just the latest question
history = [
    {"role": "user", "content": "My favourite colour is green."},
    {"role": "assistant", "content": first.message.content},  # the model's reply from Cell 15
    {"role": "user", "content": "What is my favourite colour?"},
]

reply = ollama.chat(model="gemma4:e2b", messages=history)
print(reply.message.content)

Based on what you told me earlier, your favourite colour is **green**.


### Cell 21: Plant a false reply
We change what the assistant "said" and see which one the model believes.

In [21]:
# Cell 21: Replace the models earlier reply with a made up one

#list of dictions representing the conversation history
history = [ 
    {"role":"user", "content": "My favourite colour is green."},
    {"role":"assistant", "content": "Your favourite colour is blue."},  # made up reply
    {"role":"user", "content": "What is my favourite colour?"}
]

reply = ollama.chat(
    model="gemma4:e2b",
    messages=history
)

print(reply.message.content)

Your favourite colour is green.


## Part C result: the history is the only memory

| Cell | History sent | Model answer |
|---|---|---|
| 16 | None (new call) | Does not know the colour |
| 20 | Real conversation | |
| 21 | Planted assistant reply saying red | |

**Lesson:** the model answers from the history it is given. The code that builds the history decides what the model "remembers".

## Part C result: the history is the only memory

| Cell | History sent | Model answer |
|---|---|---|
| 16 | None (new call) | Does not know the colour |
| 20 | Real conversation | Green |
| 21 | Planted assistant reply saying red | Green (the user's own words won) |

**Lesson:** the model answers from the history it is given. In this test it trusted the user's words over a planted assistant reply, but one test is not proof. The code that builds the history must be trusted.

### Cell 22: Start an empty history
This list will hold the whole conversation.

In [22]:
# Cell 22: Start an empty conversation history

chat_history = []

### Cell 23: The `ask` helper
It adds our message, sends the full history, and stores the reply.

In [23]:
# Cell 23: Helper that sends one message and keeps the full history

def ask(text):
    # Add the user's message to the shared history
    chat_history.append({"role": "user", "content": text})
    # Send the whole history so the model sees earlier turns
    response = ollama.chat(model = "gemma4:e2b", messages = chat_history)
    answer = response.message.content

    #store the models reply so the next call can see it
    chat_history.append({"role": "assistant", "content": answer})
    return answer

### Cell 24: Test the memory
We only type the new question. The helper builds the history.

In [24]:
# Cell 24: Test that the model now remembers across calls
print(ask("My favourite colour is green."))
print("---")
print(ask("What is my favourite colour?"))

Green is a wonderful colour! It's often associated with nature, growth, freshness, and tranquility. 🌿💚
---
Based on what you told me earlier, your favorite color is **green**! 💚


In [25]:
# Cell 25: Print the stored history
for message in chat_history:
    print(message["role"], ":", message["content"][:80])

user : My favourite colour is green.
assistant : Green is a wonderful colour! It's often associated with nature, growth, freshnes
user : What is my favourite colour?
assistant : Based on what you told me earlier, your favorite color is **green**! 💚


### Cell 26: Wipe the history between questions
We empty the notebook after the first question. Does the model still know the colour?

In [26]:
# Cell 26: Empty the history between two questions
chat_history = []                      # start clean
print(ask("My favourite colour is green."))
print("--- wiping the history now ---")
chat_history = []                      # wipe it
print(ask("What is my favourite colour?"))
print(len(chat_history))               # how many messages are stored now?

Green is a wonderful color! It's a very refreshing and calming shade. It reminds many people of nature and growth. 🌿💚
--- wiping the history now ---
I don't have access to your personal information, so I don't know what your favorite color is!

You'll have to tell me! 😊
2


## Part D result: the history grows by itself

| Step | `chat_history` size | What the model sees |
|---|---|---|
| Start | 0 | nothing |
| After question 1 and its reply | 2 | question 1 only (at the moment of the call) |
| After question 2 and its reply | 4 | questions and replies so far |

**Lesson:** the notebook (`chat_history`) is the only memory. Wiping it erases the memory. A longer history means more text to send each time.

## Part D result: wiping the history erases the memory

| Step | `chat_history` size | Model answer |
|---|---|---|
| Question 1 | 2 | Replies to the colour |
| History wiped | 0 | |
| Question 2 | 2 | Does not know the colour |

**Lesson:** the model has no memory of its own. The list we send is all it knows.

## Part E: Stage 4, structured output

Code cannot do much with free text. We ask the model for JSON, which code can read.

Short idea first. Until now, the model gave us free text. A person can read that, but code can’t use it. Your R&B project needs answers like “part name, quantity, price” that a program can read and add up. We ask the model for JSON, a plain-text format for organised data, and then Python reads it.

JSON looks like this:

```json
{"part": "circuit breaker", "quantity": 2}
```

### Cell 27: Ask the model for JSON
We describe the shape we want and ask for nothing else.

In [27]:
# Cell 27: Ask for one part as JSON
response = ollama.chat(
    model="gemma4:e2b",
    messages=[{"role": "user", "content": (
        "Give me one part of a desk lamp as JSON with these keys: "
        "name (text), material (text), quantity (whole number). "
        "Reply with the JSON only."
    )}],
)
raw = response.message.content
print(raw)
print(type(raw))

```json
{
  "name": "Lampshade",
  "material": "Fabric",
  "quantity": 1
}
```
<class 'str'>


### Cell 28: Turn the text into data
`json.loads` converts JSON text into a Python dictionary. Then we can read one field by its label.

In [28]:
# Cell 28: Parse the JSON text into a Python dictionary
import json

part = json.loads(raw)
print(type(part))
print(part["name"])
print(part["quantity"] + 1)   # proves the quantity is a real number

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

he model wrapped the JSON in three backticks and the word json. That is how chat apps show code blocks, and the model copied the habit. We asked for “the JSON only”, and it still added the wrapper.

How to read the error

Read the last line first:

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Part	Meaning
JSONDecodeError	Python tried to read JSON and failed
Expecting value	It expected the start of a value, like {
line 1 column 1 (char 0)	The problem is at the very first character

The first character is a backtick, not {. So Python gave up straight away. The long list of lines above it only shows the path through the package’s code, and you can skip it.

In [29]:
# Cell 29: Show the first characters of the reply, including hidden ones
print(repr(raw[:12]))

'```json\n{\n  '


### Cell 30: Ask for JSON properly
The `format="json"` option makes the server return clean JSON, so there is no code-block wrapper.

In [31]:
# Cell 30: Ask for JSON with the format option
response = ollama.chat(
    model = "gemma4:e2b",
    messages = [{"role": "user", "content": (
        "Give me one part of a desk lamp as JSON with these keys: "
        "name (text), material (text), quantity (whole number). "
    )}],
    format = "json"
)

raw = response.message.content
print(repr(raw[:12]))
print(raw)


'{\n  "name": '
{
  "name": "Lamp Shade",
  "material": "Fabric",
  "quantity": 1
}


### Cell 31: Parse the clean JSON
Same step as Cell 28, now on clean text.

In [32]:
# Cell 31: Parse the JSON text into a Python dictionary
part = json.loads(raw)
print(type(part))
print(part["name"])
print(part["quantity"] + 1)

<class 'dict'>
Lamp Shade
2


## Part E result: free text vs structured output

| Cell | What we did | Result |
|---|---|---|
| 27 | Asked for "JSON only" in the message | The reply had a code-block wrapper |
| 28 | `json.loads` on that reply | Failed: the first character was a backtick |
| 30 | Added `format="json"` | Clean JSON |
| 31 | `json.loads` on clean JSON | Worked |

**Lesson:** asking nicely in the message is not enough. A format option makes the server enforce the shape.